[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Interfaces


## What you will be able to do

Write down the methods a family of classes must provide, so that a class missing one is refused the
moment an object is created; describe what a function needs from the objects it is given, without
requiring them to share a parent; and use the standard library's own interfaces to check what an
object can do.


## The idea

### The problem

Every station in this guide has had a `report` method, and the code that used stations has called it
without asking. That worked because each class happened to define one. Nothing required it.

Now a buoy joins the network, written by someone else, who called the method `summary`. Nothing goes
wrong when the class is written, and nothing goes wrong when a buoy is created. The first sign of
trouble is an `AttributeError` at the moment something asks the buoy for its report, which may be at
the end of a nightly job, hours after the buoy's data was collected, or in a part of the program its
author never ran.

This is the cost of the flexibility the **Inheritance** notebook relied on. Code that calls
`station.report()` works on any object that has one, whatever its class, and nothing checks until the
call is made.

What is wanted is a way to write the requirement down, so that a class which does not meet it is
caught early, when an object is made, and a way to say what a function needs without forcing every
object to share a parent.

### What an interface is

> An **interface** is the set of methods that code relies on an object having. An **abstract base
> class** writes an interface down as a class: the methods it marks `@abstractmethod` have no working
> body, and Python refuses to create an object from any class, its subclasses included, until every
> one of them has been defined. A **Protocol** writes the same kind of list without anyone inheriting
> from it: a class fits a protocol simply by having the methods.

### Why it works that way

An abstract base class moves the failure earlier. The buoy's missing `report` becomes a `TypeError`
when a buoy is created, naming the class and the method, instead of an `AttributeError` somewhere
later. The base class can also carry working methods that every subclass shares, and those methods
can call the abstract ones, so each subclass fills in only the part that differs.

The catch is that it only works for classes that inherit from it. Code written by another team, or
in another library, has no reason to inherit from your `Station`. A Protocol describes a requirement
without that: any class with a `report` method fits a `Reports` protocol, whatever it inherits from.
Protocols are used mainly by type checkers, tools that read a program before it runs. When the
program runs, `isinstance` can test an object against one, but only for whether the methods exist,
not for what they take or return.

The standard library has used both ideas for years. `collections.abc` defines interfaces such as
`Iterable` and `Sized`, and a class counts as `Iterable` just by having `__iter__`, the method from
the **Context Managers and Iterators** notebook.

### Where you will meet this

Libraries that expect you to extend them often hand you an abstract base class, and the error when a
method is forgotten is exactly the one in this notebook. `isinstance(x, collections.abc.Iterable)` is
the standard way to ask whether something can be looped over, and `@abstractmethod` is the decorator
the **Decorators** notebook listed for this notebook.

### What this notebook covers

A network with no interface against one with an abstract base class, through the same numbered steps.
Then abstract methods, shared code in the base class, Protocols and what their runtime check leaves
out, `collections.abc`, and how to choose. Then one network that uses all three.

### A first look

A class that is missing the method its parent requires. There is nothing to run yet: read it, and read
the output underneath it.

```python
from abc import ABC, abstractmethod


class Station(ABC):
    @abstractmethod
    def report(self):
        """Return a one-line report."""


class Buoy(Station):
    def summary(self):
        return "Utsira: mean 6.5"


try:
    Buoy()
except TypeError as error:
    print(error)
```

```
Can't instantiate abstract class Buoy without an implementation for abstract method 'report'
```

`Buoy` was refused the moment it was created, and the message names the method it is missing.


## Setup

Three imports, bringing in seven names.

- `ABC` is the parent class that makes abstract methods enforced
- `abstractmethod` marks a method that every subclass must define
- `Protocol` describes the methods an object needs, without a parent class
- `runtime_checkable` lets `isinstance` test an object against a Protocol
- `Iterable`, `Sized` and `Sequence` are three of the standard library's own interfaces, used in one
  section

**Run this cell before the rest of the notebook.**


In [1]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable
from collections.abc import Iterable, Sized, Sequence

print("ready")


ready


## Worked examples

### Before and after: a missing method found late, or found at creation

Here is the problem from the top of this notebook, in code. Both versions go through the same three
steps, numbered in the code and in the output:

1. Create a land station and print its report.
2. Create a buoy whose author called its method `summary` instead of `report`. The mistake should be
   caught here, when the buoy is created.
3. Report on the whole network. Every station in it should produce a report.

First, the two classes with no interface: each simply has whatever methods its author wrote.


In [2]:
class LandStation:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def report(self):
        return f"{self.name}: mean {round(sum(self.readings) / len(self.readings), 2)}"


class BuoyStation:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def summary(self):
        return f"{self.name} (buoy): mean {round(sum(self.readings) / len(self.readings), 2)}"


The three steps.


In [3]:
network = []

# 1. Create a land station and print its report.
land = LandStation("Tromso", [-4.1, -2.6])
network.append(land)
print("1.", land.report())

# 2. Create a buoy whose author called its method summary instead of report.
#    The mistake should be caught here, when the buoy is created.
try:
    buoy = BuoyStation("Utsira", [6.2, 6.8])
    network.append(buoy)
    print("2. created", buoy.name)
except TypeError as error:
    print("2. refused:", error)

# 3. Report on the whole network. Every station in it should produce a report.
for station in network:
    try:
        print("3.", station.report())
    except AttributeError as error:
        print("3. failed:", error)


1. Tromso: mean -3.35
2. created Utsira
3. Tromso: mean -3.35
3. failed: 'BuoyStation' object has no attribute 'report'


Step 2 created the buoy without any trouble, so it joined the network. The mistake surfaced only in
step 3, when something finally asked the buoy for a report. Here that is a few lines later. In a real
program it can be hours later, or in code the buoy's author never ran, and the error names the missing
attribute rather than the class that should have had it.

Now the same two classes with an abstract base class above them. `Station` writes the requirement
down: every station has a `report`.


In [4]:
class Station(ABC):
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @abstractmethod
    def report(self):
        """Return a one-line report on this station."""


class LandStation(Station):
    def report(self):
        return f"{self.name}: mean {round(sum(self.readings) / len(self.readings), 2)}"


class BuoyStation(Station):
    def summary(self):
        return f"{self.name} (buoy): mean {round(sum(self.readings) / len(self.readings), 2)}"


The same three steps, with the code unchanged.


In [5]:
network = []

# 1. Create a land station and print its report.
land = LandStation("Tromso", [-4.1, -2.6])
network.append(land)
print("1.", land.report())

# 2. Create a buoy whose author called its method summary instead of report.
#    The mistake should be caught here, when the buoy is created.
try:
    buoy = BuoyStation("Utsira", [6.2, 6.8])
    network.append(buoy)
    print("2. created", buoy.name)
except TypeError as error:
    print("2. refused:", error)

# 3. Report on the whole network. Every station in it should produce a report.
for station in network:
    try:
        print("3.", station.report())
    except AttributeError as error:
        print("3. failed:", error)


1. Tromso: mean -3.35
2. refused: Can't instantiate abstract class BuoyStation without an implementation for abstract method 'report'
3. Tromso: mean -3.35


The buoy was refused in step 2, at creation, with a message that names both the class and the method
it lacks. It never joined the network, so step 3 reported on the one station that could.

| | No interface | Abstract base class |
|---|---|---|
| When the missing `report` was found | step 3, when a report was needed | step 2, when the buoy was created |
| What the error names | the attribute that was looked up | the class, and the method it is missing |
| What can be created | any class | only a class that defines every abstract method |
| Where the requirement is written | nowhere; it is implied by the calls | in `Station`, as `@abstractmethod` |

The rest of this notebook takes the idea apart, and then goes past it.

| Question | The section that answers it |
|---|---|
| What does `@abstractmethod` actually do? | Abstract methods |
| Can the base class still share working code? | An abstract base class can share code |
| What about objects that do not inherit from `Station`? | Protocols: requiring methods without a parent |
| What does a runtime check against a Protocol leave out? | What the runtime check leaves out |
| How does the standard library do this? | `collections.abc`: the standard library's interfaces |
| Which one should I use? | Choosing between them |

### Abstract methods

`@abstractmethod` marks a method as required, and inheriting from `ABC` is what makes Python enforce
it. The base class itself has the abstract method undefined, so it cannot be created either.


In [6]:
try:
    Station("Nowhere", [])
except TypeError as error:
    print("Station itself:", error)

print("Station still needs:    ", Station.__abstractmethods__)
print("LandStation still needs:", LandStation.__abstractmethods__)
print("a land station is a Station:", isinstance(LandStation("Tromso", [-4.1]), Station))


Station itself: Can't instantiate abstract class Station without an implementation for abstract method 'report'
Station still needs:     frozenset({'report'})
LandStation still needs: frozenset()
a land station is a Station: True


`__abstractmethods__` is the list Python checks when an object is created. `Station` has one entry,
`report`, and `LandStation` has none, because it defined `report`. An object can be created only when
that list is empty.

A land station is still a `Station`, so `isinstance` and everything from the **Inheritance** notebook
applies.

### An abstract base class can share code

The base class is not limited to requirements. It can hold working methods too, and those methods can
call the abstract ones. Here `Station` writes the whole report, and each subclass supplies only a
description of itself.


In [7]:
class Station(ABC):
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name} ({self.describe()}): mean {self.mean()}"

    @abstractmethod
    def describe(self):
        """Say what kind of station this is."""


class LandStation(Station):
    def describe(self):
        return "on land"


class BuoyStation(Station):
    def __init__(self, name, readings, depth):
        super().__init__(name, readings)
        self.depth = depth

    def describe(self):
        return f"buoy, sensor at {self.depth} m"


for station in [LandStation("Tromso", [-4.1, -2.6]), BuoyStation("Utsira", [6.2, 6.8], depth=2)]:
    print(station.report())


Tromso (on land): mean -3.35
Utsira (buoy, sensor at 2 m): mean 6.5


`report` and `mean` are written once, in `Station`. `report` calls `self.describe()`, which `Station`
does not define, and each subclass supplies it. The abstract method is how `Station` says "this part
is yours", and Python makes sure every subclass writes it.

### Protocols: requiring methods without a parent

An abstract base class only helps with classes that inherit from it. Suppose the network also takes
reports from a satellite, written by another team, whose class inherits from nothing of yours.

A Protocol lists what is needed without asking anyone to inherit from it. `@runtime_checkable` lets
`isinstance` test an object against it while the program runs.


In [8]:
@runtime_checkable
class Reports(Protocol):
    def report(self) -> str:
        ...


class Satellite:
    """Another team's class. It has a report method and no parent of ours."""

    def report(self):
        return "satellite pass over Tromso: clear sky"


class Thermometer:
    def read(self):
        return -4.1


for thing in [LandStation("Tromso", [-4.1, -2.6]), Satellite(), Thermometer()]:
    print(f"{type(thing).__name__:<12} fits Reports: {isinstance(thing, Reports)}")

print()
print("Satellite's parents:", [c.__name__ for c in Satellite.__mro__])


LandStation  fits Reports: True
Satellite    fits Reports: True
Thermometer  fits Reports: False

Satellite's parents: ['Satellite', 'object']


`Satellite` inherits from `object` and nothing else, and it fits `Reports`, because it has a `report`
method. `Thermometer` does not. Nobody had to change their class to fit: the protocol describes, it
does not require.

`-> str` after the parameters is a return annotation, saying what `report` should give back, and
`...` is a placeholder body, since a protocol method is a description and never runs.

### What the runtime check leaves out

`isinstance` against a Protocol asks one question: does the object have attributes with these names?
It does not ask what they are.


In [9]:
class NumberNamedReport:
    report = 42


class WrongSignature:
    def report(self, verbose):
        return "..."


for thing in [NumberNamedReport(), WrongSignature()]:
    print(f"{type(thing).__name__:<17} fits Reports: {isinstance(thing, Reports)}")
    try:
        thing.report()
    except TypeError as error:
        print(f"{'':<17} but report() raises: {error}")


NumberNamedReport fits Reports: True
                  but report() raises: 'int' object is not callable
WrongSignature    fits Reports: True
                  but report() raises: WrongSignature.report() missing 1 required positional argument: 'verbose'


Both fit, and neither works. A `report` that is a number, and a `report` that needs an argument nobody
will pass, both satisfy a check that looks only at names. The types in the protocol, `-> str`, are
there for type checkers, which read the annotations before a program runs. When the program runs,
treat `isinstance` against a Protocol as "has these names", and nothing more.

### `collections.abc`: the standard library's interfaces

`collections.abc` holds interfaces the standard library uses everywhere. Several of them work like a
protocol: a class counts as `Iterable` if it has `__iter__`, and as `Sized` if it has `__len__`, with
no inheritance. The `Readings` class from the **Composition over Inheritance** notebook had both.


In [10]:
class Readings:
    def __init__(self):
        self._values = []

    def append(self, value):
        self._values.append(value)

    def __len__(self):
        return len(self._values)

    def __iter__(self):
        return iter(self._values)


readings = Readings()
readings.append(-4.1)

print("Readings is Iterable:", isinstance(readings, Iterable))
print("Readings is Sized:   ", isinstance(readings, Sized))
print("its parents:         ", [c.__name__ for c in Readings.__mro__])
print("a list is Iterable:  ", isinstance([], Iterable))
print("an int is Iterable:  ", isinstance(3, Iterable))


Readings is Iterable: True
Readings is Sized:    True
its parents:          ['Readings', 'object']
a list is Iterable:   True
an int is Iterable:   False


`Readings` inherits from nothing, and the standard library still recognizes it as something that can
be looped over and measured, because it has the two methods that make that true.

Others work like an abstract base class, and give working methods in return for the required ones.
Inheriting from `Sequence` and writing `__getitem__` and `__len__` supplies the rest of what a
read-only sequence can do.


In [11]:
class History(Sequence):
    def __init__(self, values):
        self._values = list(values)

    def __getitem__(self, index):
        return self._values[index]

    def __len__(self):
        return len(self._values)


history = History([-4.1, -2.6, -3.8, -2.6])

print("-2.6 in history:    ", -2.6 in history)
print("history.index(-3.8):", history.index(-3.8))
print("history.count(-2.6):", history.count(-2.6))
print("reversed:           ", list(reversed(history)))


class Unfinished(Sequence):
    def __getitem__(self, index):
        return 0


try:
    Unfinished()
except TypeError as error:
    print()
    print("Unfinished:", error)


-2.6 in history:     True
history.index(-3.8): 2
history.count(-2.6): 2
reversed:            [-2.6, -3.8, -2.6, -4.1]

Unfinished: Can't instantiate abstract class Unfinished without an implementation for abstract method '__len__'


`History` wrote two methods and got `in`, `index`, `count` and `reversed` from `Sequence`, which
builds all of them out of `__getitem__` and `__len__`. `Unfinished` left out `__len__`, and `Sequence`
refused it exactly as `Station` refused the buoy.

### Choosing between them

| You want | Use |
|---|---|
| a family of classes you write, each required to provide certain methods, with shared code | an abstract base class |
| to accept objects from anywhere, as long as they have the right methods | a Protocol |
| to ask whether something can be looped over, measured or indexed | `collections.abc` |
| a small program where every class is in front of you | nothing extra: call the method |

The last row is a real option. An interface earns its place when classes are written by different
people, or at different times, and a missing method would otherwise be found late.

### Putting it together: a network that says what it needs

An abstract base class for the stations this program owns, with shared code and one required method.
A runtime-checkable Protocol for anything that can report, including a satellite from another team.
And one loop over a mixed feed that reports on everything it can, and skips what it cannot.


In [12]:
class Station(ABC):
    """What every station in the network must be able to do."""

    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"

    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name} ({self.describe()}): mean {self.mean()}"

    @abstractmethod
    def describe(self):
        """Say what kind of station this is."""


class LandStation(Station):
    def describe(self):
        return "on land"


class BuoyStation(Station):
    def __init__(self, name, readings, depth):
        super().__init__(name, readings)
        self.depth = depth

    def describe(self):
        return f"buoy, sensor at {self.depth} m"


feed = [
    LandStation("Tromso", [-4.1, -2.6]),
    BuoyStation("Utsira", [6.2, 6.8], depth=2),
    Satellite(),
    Thermometer(),
]

for item in feed:
    if isinstance(item, Reports):
        print(item.report())
    else:
        print("skipped", type(item).__name__, "because it cannot report")

print()
print("the stations among them:", [item for item in feed if isinstance(item, Station)])


Tromso (on land): mean -3.35
Utsira (buoy, sensor at 2 m): mean 6.5
satellite pass over Tromso: clear sky
skipped Thermometer because it cannot report

the stations among them: [LandStation('Tromso'), BuoyStation('Utsira')]


The loop asked each item one question, whether it fits `Reports`, and the land station, the buoy and
the satellite all did, by different routes: two inherit `report` from `Station`, and the satellite
simply has one. The thermometer was skipped instead of crashing the loop. The second question,
`isinstance(item, Station)`, picked out only the classes this program owns.

A station class that forgets the method it is required to write never gets that far.


In [13]:
class Unfinished(Station):
    pass


try:
    Unfinished("Draft", [1.0])
except TypeError as error:
    print(error)


Can't instantiate abstract class Unfinished without an implementation for abstract method 'describe'


### Where each part came from

| In the network | What it relies on | The section that showed it |
|---|---|---|
| `Unfinished` refused at creation | `@abstractmethod` on a class that inherits from `ABC` | Abstract methods |
| `report` and `mean` written once, in `Station` | an abstract base class can hold working code | An abstract base class can share code |
| each subclass writing only `describe` | shared code calling an abstract method | An abstract base class can share code |
| the satellite reporting with no parent of ours | a Protocol describes, and does not require inheriting | Protocols: requiring methods without a parent |
| the thermometer skipped | `isinstance` against a runtime-checkable Protocol | Protocols: requiring methods without a parent |
| trusting the check only for names | the runtime check looks at names, not types | What the runtime check leaves out |
| picking out the stations the program owns | `isinstance` with the abstract base class | Abstract methods |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/12-interfaces-solutions.ipynb).

**1.** Write an abstract base class `Instrument` with an abstract method `read()`. Show that
`Instrument()` is refused, and print the error.


In [14]:
# your code here


**2.** Write `Thermometer(Instrument)`, whose `read()` returns a number. Create one and read it.


In [15]:
# your code here


**3.** Give `Instrument` a working method `describe()` that uses `self.read()`, returning something
like `"Thermometer reads -4.1"`. Call it on a thermometer.


In [16]:
# your code here


**4.** Write `Barometer(Instrument)` with no `read()`, try to create one, and print the error. Say in a
comment what the message tells you.


In [17]:
# your code here


**5.** Write a runtime-checkable Protocol `Readable` that requires `read()`. Show that `isinstance` is
`True` for a thermometer, `True` for an unrelated class that has a `read()` method and inherits from
nothing, and `False` for a string.


In [18]:
# your code here


**6.** Write a class that holds a list and has `__len__` and `__iter__`, inheriting from nothing. Show
that it counts as `Sized` and `Iterable`, then show that a class without `__iter__` is not `Iterable`.


In [19]:
# your code here


## Common errors

### TypeError: creating the abstract base class itself

An abstract base class describes what its subclasses must do. It is not meant to be used directly, and
Python refuses to create one.


In [20]:
Station("Nowhere", [])


TypeError: Can't instantiate abstract class Station without an implementation for abstract method 'describe'

The message has the same form as the buoy's, and names `describe`, the one method this `Station`
leaves for its subclasses to write. Create a subclass that defines every abstract method, and create
that instead.

### TypeError: a subclass that defines only some of the abstract methods

When a base class requires more than one method, the error lists every one still missing.


In [21]:
class Calibrated(ABC):
    @abstractmethod
    def describe(self):
        """Say what kind of instrument this is."""

    @abstractmethod
    def calibrate(self):
        """Adjust the instrument against a reference."""


class HalfDone(Calibrated):
    def describe(self):
        return "thermometer"


HalfDone()


TypeError: Can't instantiate abstract class HalfDone without an implementation for abstract method 'calibrate'

`HalfDone` defined `describe` and forgot `calibrate`, and the message names `calibrate` alone. Read
the end of the message first: it is the list of what is left to write.

### TypeError: `isinstance` with a Protocol that is not runtime checkable

A Protocol without `@runtime_checkable` is a description for type checkers only, and `isinstance`
refuses to use it.


In [22]:
class Describes(Protocol):
    def describe(self) -> str:
        ...


isinstance(LandStation("Tromso", [-4.1]), Describes)


TypeError: Instance and class checks can only be used with @runtime_checkable protocols

The message names the fix. Add `@runtime_checkable` above the class, and remember what the check will
then do: look for the names, and nothing more.

### The quiet one: `@abstractmethod` without `ABC`

The decorator marks a method as abstract. Only a class that inherits from `ABC` acts on the mark.


In [23]:
class NotEnforced:
    @abstractmethod
    def report(self):
        """Return a one-line report on this station."""


class Buoy(NotEnforced):
    pass


buoy = Buoy()

print("created anyway:", type(buoy).__name__)
print("buoy.report() returns:", repr(buoy.report()))


created anyway: Buoy
buoy.report() returns: None


No error. `Buoy` was created without a `report` of its own, and calling `report` ran the base class's
placeholder, which has only a docstring and so returns `None`. The requirement the author wrote down
is not being enforced, and the class looks as though it is.

The fix is one word, `class NotEnforced(ABC):`. With it, `Buoy()` raises the `TypeError` from the
start of this notebook.


## Recap

- Calling a method an object does not have fails only when the call happens, which can be far from the
  mistake.
- An abstract base class inherits from `ABC` and marks the methods it requires with `@abstractmethod`.
- Python refuses to create an object from a class that leaves any abstract method undefined, and names
  the ones missing.
- The abstract base class itself cannot be created either.
- An abstract base class can hold working methods, including ones that call the abstract ones.
- `@abstractmethod` does nothing unless the class inherits from `ABC`.
- A Protocol lists the methods an object needs, and any class that has them fits, whatever its parents.
- `isinstance` works with a Protocol only when it is marked `@runtime_checkable`.
- The runtime check sees only that the names exist, not what they take or return.
- In `collections.abc`, `Iterable` is anything with `__iter__` and `Sized` anything with `__len__`.
- Inheriting from `Sequence` and writing `__getitem__` and `__len__` also gives `in`, `index`, `count`
  and `reversed`.
- Use an abstract base class for a family you write, a Protocol for objects from anywhere, and nothing
  extra when the program is small.


## What is next

The **Exceptions as Classes** notebook. Every exception you have caught is a class in a hierarchy, and
`ReadingError(ValueError)` in the **Composition over Inheritance** notebook was the first one you
wrote. That notebook builds a small family of your own, decides what information each exception should
carry, and shows why `except` clauses depend on getting the hierarchy right.


---

&#8592; **Previous:** [Dataclasses](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/11-dataclasses.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Exceptions as Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/13-exceptions-as-classes.ipynb) &#8594;
